# TradoVera — Tutoriel 2 : Simulation (Backtesting) d'une stratégie simple

Ce notebook implémente une stratégie quantitative classique de croisement de moyennes mobiles (SMA Crossover 50/200) en utilisant les cours quotidiens ajustés stockés dans le Data Lake.

In [ ]:
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath(".."))
import tvdata_sdk

### 1. Charger les données historiques ajustées
Pour le backtesting, il est capital d'utiliser des prix ajustés (`adjusted=True`) pour refléter les splits d'actions et les dividendes réinvestis.

In [ ]:
symbol = "AAPL"
df = tvdata_sdk.get_ohlcv(symbol, timeframe="D1")
df = df.sort_values("timestamp").reset_index(drop=True)
df["date"] = pd.to_datetime(df["timestamp"])
df.head()

### 2. Calculer les indicateurs techniques
Calculons les moyennes mobiles simples (SMA) sur 50 et 200 jours :

In [ ]:
df["SMA50"] = df["close"].rolling(window=50).mean()
df["SMA200"] = df["close"].rolling(window=200).mean()
df = df.dropna().reset_index(drop=True)
df[["date", "close", "SMA50", "SMA200"]].tail()

### 3. Modéliser la stratégie de trading
- **Signal d'achat (Golden Cross)** : La SMA 50 passe au-dessus de la SMA 200 (on achète).
- **Signal de vente (Death Cross)** : La SMA 50 passe en-dessous de la SMA 200 (on vend).

In [ ]:
df["Signal"] = 0.0
df.loc[df["SMA50"] > df["SMA200"], "Signal"] = 1.0
df["Position"] = df["Signal"].diff()

# Calculer les rendements quotidiens
df["Market_Returns"] = df["close"].pct_change()
df["Strategy_Returns"] = df["Market_Returns"] * df["Signal"].shift(1)

# Rendements cumulés
df["Cum_Market"] = (1 + df["Market_Returns"].fillna(0)).cumprod() - 1
df["Cum_Strategy"] = (1 + df["Strategy_Returns"].fillna(0)).cumprod() - 1

print(f"Rendement final du Marché : {df['Cum_Market'].iloc[-1] * 100:.2f}%")
print(f"Rendement final de la Stratégie : {df['Cum_Strategy'].iloc[-1] * 100:.2f}%")